# Lists

**Topic:** `3.1` &nbsp;|&nbsp; **Module 3: Core Data Structures**

*Reference implementations for all 10 exercises plus the robotics challenge.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M3)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [03_data_structures/README.md](../README.md)

> **Compare, do not copy**
>
> Work each exercise yourself first. When you open this notebook, compare your approach with the reference: check the reasoning, the edge cases and the complexity. If your solution differs, that is fine — verify it with the tests provided and keep whichever is clearer.

## Solution 1 — Clamp readings without touching the caller's list

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** copying, append, range clamping, immutability of inputs

**Explanation and reasoning**

Copying is the contract: nothing in the function writes to the argument, and clamped starts life as a private list. The bool guard precedes the numeric check because bool subclasses int in Python - without it, True would clamp to 1.0 and silently pollute the log.

In [ ]:
def clamp_readings(readings, low=0.0, high=1.0):
    """Return a clamped copy; the input list is never modified."""
    if low > high:
        raise ValueError('low must not exceed high')
    clamped = []
    for value in readings:
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            continue
        clamped.append(min(max(value, low), high))
    return clamped

**Complexity**

Time O(n); space O(k) for the k numeric values kept.

**Edge cases and failure modes**

Empty input returns []. low > high raises. bools are skipped, not clamped.

**Alternative approaches**

A list comprehension with an inline guard is shorter but buries the validation policy.

**Tests — run the cell to verify the reference solution**

In [ ]:
src = [0.4, 1.3, 'x', -0.2]
out = clamp_readings(src)
assert out == [0.4, 1.0, 0.0]
assert src == [0.4, 1.3, 'x', -0.2]

---

## Solution 2 — Rotate a command list with slicing

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** slicing, rotation, modulo indexing

**Explanation and reasoning**

Modulo normalises any k - negative or huge - into [0, len), so the slice arithmetic only ever sees a valid offset. The tail slice commands[-k:] is the part that wraps, and concatenating the two slices builds a fresh list, leaving the original untouched.

In [ ]:
def rotate(commands, k):
    """Rotate right by k with wrap-around; returns a new list."""
    if not commands:
        return []
    k = k % len(commands)
    if k == 0:
        return list(commands)
    return commands[-k:] + commands[:-k]

**Complexity**

Time O(n); space O(n) for the new list.

**Edge cases and failure modes**

Empty list returns []. k = 0 returns a copy via the early return plus the modulo branch.

**Alternative approaches**

collections.deque.rotated exists in 3.2+ but hides the slicing lesson.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert rotate(['a', 'b', 'c', 'd'], 2) == ['c', 'd', 'a', 'b']
assert rotate([1, 2, 3], 0) == [1, 2, 3]
assert rotate([1, 2, 3], -1) == [2, 3, 1]

---

## Solution 3 — Running totals for a battery drain log

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** accumulation, append, zip

**Explanation and reasoning**

The accumulator starts at zero before the loop, so the first append is exactly deltas[0] and every later append adds one more delta. Building the result with append in the same pass keeps the whole computation linear - a naive nested-sum version would be quadratic.

In [ ]:
def running_totals(deltas):
    """Cumulative sums: element i is the sum of deltas[0..i]."""
    total = 0.0
    totals = []
    for delta in deltas:
        total += delta
        totals.append(total)
    return totals

**Complexity**

Time O(n); space O(n).

**Edge cases and failure modes**

Empty input never enters the loop and returns []. A single delta returns [delta].

**Alternative approaches**

itertools.accumulate does this in C, but the explicit loop shows the invariant.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert running_totals([1.0, -0.5, 2.0]) == [1.0, 0.5, 2.5]
assert running_totals([]) == []
assert running_totals([-1, -2]) == [-1, -3]

---

## Solution 4 — Batch a telemetry stream into fixed-size chunks

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** slicing, range with step, batching

**Explanation and reasoning**

range(0, len(values), size) yields exactly the start offsets 0, size, 2*size, ... while the last start is below len(values); slicing from each start clamps automatically at the end of the sequence, so the short final chunk falls out without a special case.

In [ ]:
def batch(values, size):
    """Split into consecutive chunks of at most size items."""
    if size < 1:
        raise ValueError('size must be at least 1')
    return [values[i:i + size] for i in range(0, len(values), size)]

**Complexity**

Time O(n); space O(n) across all chunks.

**Edge cases and failure modes**

Empty input yields []. size greater than len(values) yields one chunk equal to a copy of the input.

**Alternative approaches**

A while loop with pop(0) destroys the input; a deque-based splitter trades clarity.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert batch([1, 2, 3, 4, 5], 2) == [[1, 2], [3, 4], [5]]
assert batch([], 3) == []
flat = [x for chunk in batch(list(range(10)), 3) for x in chunk]
assert flat == list(range(10))

---

## Solution 5 — Split evens and odds while keeping their order

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** conditional append, stability, two accumulators

**Explanation and reasoning**

One pass, two accumulators, one conditional append - relative order within each side is preserved because appends happen in input order. Python's modulo gives -3 % 2 == 1, so negative odds land on the odds side without special-casing the sign.

In [ ]:
def split_parity(values):
    """Split into (evens, odds), each preserving input order."""
    evens, odds = [], []
    for value in values:
        (evens if value % 2 == 0 else odds).append(value)
    return evens, odds

**Complexity**

Time O(n); space O(n).

**Edge cases and failure modes**

Empty input returns ([], []). All-even input puts a copy of everything in evens.

**Alternative approaches**

Two comprehensions are shorter but scan the input twice.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert split_parity([1, 2, 3, 4, 5, 6]) == ([2, 4, 6], [1, 3, 5])
assert split_parity([]) == ([], [])
assert split_parity([-3, -2]) == ([-2], [-3])

---

## Solution 6 — Merge two sorted lists in linear time

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** two-pointer merge, sorted input, amortised append

**Explanation and reasoning**

Two pointers walk the inputs once, each element is appended exactly once, and the leftover tail of whichever input still has elements is spliced with extend. Using <= on the left comparison makes ties come from left, which is what makes the merge stable.

In [ ]:
def merge_sorted(left, right):
    """Stable merge of two ascending lists in O(n + m)."""
    merged = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])   # <= keeps left-first stability
            i += 1
        else:
            merged.append(right[j])
            j += 1
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged

**Complexity**

Time O(n + m); space O(n + m).

**Edge cases and failure modes**

Either input empty reduces the loop to a single extend. Duplicates are all retained.

**Alternative approaches**

heapq.merge is lazy and allocation-free per step, but returns an iterator.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert merge_sorted([1, 3, 5], [2, 4]) == [1, 2, 3, 4, 5]
assert merge_sorted([], [1]) == [1]
assert merge_sorted([1, 1], [1]) == [1, 1, 1]

---

## Solution 7 — Transpose a rectangular matrix into a new one

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** nested lists, index swapping, validation, new-list construction

**Explanation and reasoning**

Validating every row against the first row's width before any work happens turns zip's silent truncation on ragged input into an explicit failure. The nested comprehension then builds each column as a fresh list, so neither the rows nor the cells of the input are ever shared with the result.

In [ ]:
def transpose(matrix):
    """Return a new transposed matrix; ragged input raises."""
    if not matrix:
        return []
    width = len(matrix[0])
    if any(len(row) != width for row in matrix):
        raise ValueError('ragged matrix: rows differ in length')
    return [[matrix[r][c] for r in range(len(matrix))] for c in range(width)]

**Complexity**

Time O(rows * cols); space O(rows * cols) for the result.

**Edge cases and failure modes**

[].  [[]].  A single row becomes a single column. [[1, 2], [3]] raises ValueError.

**Alternative approaches**

zip(*matrix) is one line but returns tuples and truncates ragged rows silently.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert transpose([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]]
assert transpose([]) == []
try:
    transpose([[1, 2], [3]])
    raise AssertionError('should have raised')
except ValueError:
    pass

---

## Solution 8 — Rotate a list in place with O(1) extra memory

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** in-place algorithm, reversal, index arithmetic

**Explanation and reasoning**

Three reversals compose a rotation: reversing everything brings the rotating tail to the front but backwards, and reversing each half puts both halves back in order. Only index swaps occur, so memory stays constant regardless of list length. The modulo first makes the algorithm safe for any k.

In [ ]:
def rotate_in_place(values, k):
    """Rotate right by k using reversal - O(1) extra memory."""
    n = len(values)
    if n < 2:
        return None
    k %= n
    if k == 0:
        return None

    def reverse(lo, hi):
        while lo < hi:
            values[lo], values[hi] = values[hi], values[lo]
            lo, hi = lo + 1, hi - 1

    reverse(0, n - 1)      # [5, 4, 3, 2, 1]
    reverse(0, k - 1)      # [4, 5, 3, 2, 1]
    reverse(k, n - 1)      # [4, 5, 1, 2, 3]

**Complexity**

Time O(n) (each element moves twice); space O(1).

**Edge cases and failure modes**

n < 2 or k % n == 0 are no-ops. Negative k works after modulo.

**Alternative approaches**

Cycling via juggling is also O(1) memory but far harder to prove correct.

**Tests — run the cell to verify the reference solution**

In [ ]:
xs = [1, 2, 3, 4, 5]
rotate_in_place(xs, 2)
assert xs == [4, 5, 1, 2, 3]
ys = [1, 2, 3]
rotate_in_place(ys, 3)
assert ys == [1, 2, 3]

---

## Solution 9 — Sliding windows with a custom step

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** slicing, range strides, window arithmetic

**Explanation and reasoning**

The last legal start is len(values) - size; range stops before exceeding it, so incomplete windows are impossible by construction rather than by a filter. Each window is materialised as its own list so callers cannot mutate the source through a shared slice.

In [ ]:
def windows(values, size, step=1):
    """Every size-wide window, taken every step positions."""
    if size < 1 or step < 1:
        raise ValueError('size and step must be at least 1')
    starts = range(0, len(values) - size + 1, step)
    return [list(values[s:s + size]) for s in starts]

**Complexity**

Time O(num_windows * size); space O(num_windows * size).

**Edge cases and failure modes**

size > len(values) makes the range empty, returning []. Empty input returns [].

**Alternative approaches**

itertools.islice over indices streams, but the list form is simpler to test.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert windows([1, 2, 3, 4, 5], 3, step=2) == [[1, 2, 3], [3, 4, 5]]
assert windows([1, 2], 5) == []
assert windows([], 2) == []

---

## Solution 10 — Run-length encode a command stream

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** run-length encoding, run detection, tuples in lists

**Explanation and reasoning**

The loop compares each value against the current run's value and only closes the run on a change, so equal values separated by a different value correctly form two runs. The final append after the loop flushes the last run - the classic off-by-one to forget, and the reason the all-identical case must be tested.

In [ ]:
def run_length(values):
    """Encode consecutive runs as (value, count) tuples."""
    if not values:
        return []
    runs = []
    current, count = values[0], 1
    for value in values[1:]:
        if value == current:
            count += 1
        else:
            runs.append((current, count))
            current, count = value, 1
    runs.append((current, count))
    return runs

**Complexity**

Time O(n); space O(number of runs).

**Edge cases and failure modes**

Empty input returns []. A single value yields [(value, 1)]. All-identical yields one tuple.

**Alternative approaches**

itertools.groupby is equivalent but hides the grouping you are being asked to implement.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert run_length(['go', 'go', 'hold', 'go']) == [('go', 2), ('hold', 1), ('go', 1)]
assert run_length([]) == []
total = sum(count for _, count in run_length([1, 1, 2]))
assert total == 3

---

## Reflection

For each exercise where your solution differed from the reference, write one sentence explaining *why* yours was better or worse. This is the highest-value part of the notebook:

- A case the reference handles but mine did not.
- A case mine handles better — and what the reference is missing.
- A performance difference that matters at robot scale.